## Prelude

In [ ]:
import librosa
import numpy as np
import scipy.signal
from scipy.signal import butter, lfilter, freqz
import soundfile as sf

import mido

import bokeh.io
import bokeh.plotting
import bokeh.models
import bokeh.palettes
bokeh.io.output_notebook()

from IPython.display import Audio
import matplotlib.pyplot as plt

In [ ]:
BALAFON_TUNING = list(reversed([1385, 1122, 985, 827, 780, 672, 570, 496, 414, 388, 338, 282, 245, 205, 193, 166, 141, 121, 102, 95, 80]))
MIDI_NOTE_NUMS = [36, 38, 40, 41, 43,  48, 50, 52, 53, 55,  60, 62, 64, 65, 67,  72, 74, 76, 77, 79, 84]    # 84 isn't included in the sfz
N = len(BALAFON_TUNING)

## Create a rough transcription for alignment

In [ ]:
def twos_complement(hexstr, bits):
    """ Convert a hexadecimal string to a signed integer using two's complement representation. """
    value = int(hexstr, 16)
    if value & (1 << (bits - 1)):
        value -= 1 << bits
    return value

def read_sensor_data_hex(filename, v=False):
  """ Reads sensor data from a text file and returns the data in a structured format. Skips malformed rows. """
  # times, x, y, z
  data = [([], [], [], []) for _ in range(N)]
  interrupt_timestamps = [[] for _ in range(N)]

  with open(filename, "r") as f:
    for row in f:
      values = row[:-1].split(",")
      if(len(values) != 6):
        if v:
          print("Wrong number of values")
          print(row)
        continue

      timestamp = int(values[0], 16)
      sensor_number = int(values[1], 16)
      x_accel = twos_complement(values[2], 32)
      y_accel = twos_complement(values[3], 32)
      z_accel = twos_complement(values[4], 32)
      interrupt = int(values[5], 16)

      for val in [x_accel, y_accel, z_accel]:
        if (abs(val) > 40000):
          if v:
            print("Out of range")
            print(row[:-1])
          continue

      last_timestamp = None
      if (last_timestamp and timestamp <= last_timestamp):
        if v:
          print("Out of order")
          print(row[:-1])
        continue
      else:
        last_timestamp = timestamp

      if (sensor_number >= N and sensor_number < 0):
        if v:
          print(row[:-1])
        continue

      sensor_data = data[sensor_number]
      sensor_data[0].append(timestamp / 1e6)  # convert to seconds 
      sensor_data[1].append(x_accel)
      sensor_data[2].append(y_accel)
      sensor_data[3].append(z_accel)

      if (interrupt != 0):
        interrupt_timestamps[sensor_number].append(interrupt)

  return data, interrupt_timestamps

def window_rms(a, window_size):
  """ 
  Calculate the root mean square (RMS) of a signal using a sliding window.
  - a: numpy array, the input signal.
  - window_size: int, the size of the sliding window.
  Returns: numpy array, the RMS values.
  """
  a2 = np.power(a,2)
  window = np.ones(window_size)/float(window_size)
  return np.sqrt(np.convolve(a2, window, 'valid'))   # same keeps border effects

def find_peaks(data, threshold=0.3):
  """ Find peaks in the data using librosa's peak picking function, filtered by a threshold.
  - data: numpy array, the input signal.
  - threshold: float, the threshold for peak detection.
  Returns:
  - valid_peaks: list of indices where peaks are found above the threshold.
  """
  peaks = librosa.util.peak_pick(data, pre_max=100, post_max=100, pre_avg=1000, post_avg=1000, delta=0.1, wait=20)
  
  valid_peaks = []
  for i in peaks:
      if data[i] > threshold:
          valid_peaks.append(i)
  return valid_peaks

def detect_outliers(data, method="zscore", threshold=3, v=False):
  """
  Detect outliers in the data using Z-score or IQR and print information about them.
  - data: numpy array, the input data.
  - method: str, the method to use for outlier detection ("zscore" or "iqr").
  - threshold: float, the threshold for detecting outliers.

  Returns:
  - mask: numpy array, a boolean mask where True indicates non-outliers.
  """
  data = np.array(data)
  if method == "zscore":
      # Z-score method
      mean = np.mean(data)
      std = np.std(data)
      z_scores = (data - mean) / std
      mask = np.abs(z_scores) < threshold
      outliers = np.where(~mask)[0]
      if v:
        print(f"Z-score Outliers Detected: {len(outliers)}")
        for idx in outliers:
            print(f"Index: {idx}, Value: {data[idx]}, Z-score: {z_scores[idx]}")
  elif method == "iqr":
      # IQR method
      q1 = np.percentile(data, 25)
      q3 = np.percentile(data, 75)
      iqr = q3 - q1
      lower_bound = q1 - threshold * iqr
      upper_bound = q3 + threshold * iqr
      mask = (data >= lower_bound) & (data <= upper_bound)
      outliers = np.where(~mask)[0]
      if v: 
        print(f"IQR Outliers Detected: {len(outliers)}")
        for idx in outliers:
            print(f"Index: {idx}, Value: {data[idx]}, Bounds: ({lower_bound}, {upper_bound})")
  else:
      raise ValueError("Invalid method. Use 'zscore' or 'iqr'.")
  
  return mask

def sensor_data_transformations(data, methods=[], outlier_method="zscore", 
                                  outlier_threshold=100, attack_time_ms=5, release_time_ms=500, v=False):
  """  
  Transform sensor data into various formats and detect outliers.
  - data: list of lists, where each sublist contains [timestamps, x, y, z] for a sensor.
  - methods: list of strings, the transformation methods to apply (e.g., "raw_x", "raw_y", "raw_z", "magnitude", "rms", "envelope").
  - outlier_method: str, the method to use for outlier detection ("zscore" or "iqr").
  - outlier_threshold: float, the threshold for detecting outliers.
  - attack_time_ms: int, attack time in milliseconds for envelope follower.
  - release_time_ms: int, release time in milliseconds for envelope follower.
  - v: bool, verbose output.
  
  Returns: list of tuples, each containing (timestamps, transformed_signal) for each method.
  """
  transformed_signals = []
  mag = np.sqrt(np.array(data[1])**2 + np.array(data[2])**2 + np.array(data[3])**2)
  mask = detect_outliers(mag, method=outlier_method, threshold=outlier_threshold, v=v)
  mag_cleaned = mag[mask]
  timestamps_cleaned = np.array(data[0])[mask]
  x_cleaned = np.array(data[1], dtype=np.double)[mask]
  y_cleaned = np.array(data[2], dtype=np.double)[mask]
  z_cleaned = np.array(data[3], dtype=np.double)[mask]

  for method in methods:
    if method == "raw_x":
      transformed_signals.append((timestamps_cleaned, x_cleaned))
    elif method == "raw_y":
      transformed_signals.append((timestamps_cleaned, y_cleaned))
    elif method == "raw_z":
      transformed_signals.append((timestamps_cleaned, z_cleaned))
    elif method == "magnitude":
      transformed_signals.append((timestamps_cleaned, mag_cleaned))
    elif method == "rms":
      rms = window_rms(mag_cleaned, 64)
      rms -= np.median(rms)
      rms /= np.max(rms)
      timestamps_cleaned = timestamps_cleaned[:len(rms)]  # Adjust timestamps to match RMS length
      transformed_signals.append((timestamps_cleaned, rms))
    elif method == "envelope":
      # estimate sample rate
      sample_rate = 1 / (np.mean(np.diff(timestamps_cleaned)) / 1e6)
      # center and normalize z axis
      z_cleaned -= np.mean(z_cleaned)
      z_cleaned /= np.max(np.abs(z_cleaned))
      # Apply envelope follower
      # defaults: 5, 50
      transformed_signals.append((
        timestamps_cleaned,
        envelope_follower(z_cleaned, sample_rate, attack_time_ms=attack_time_ms, release_time_ms=release_time_ms)))
  return transformed_signals

def transform_data_and_find_peaks(data, 
                                  threshold=0.3, 
                                  methods=["rms"], 
                                  outlier_method="zscore", 
                                  outlier_threshold=100, 
                                  attack_time_ms=5, 
                                  release_time_ms=500,
                                  v=False):
    """ Transform sensor data and find peaks in the transformed signal.
    - data: list of lists, where each sublist contains [timestamps, x, y, z] for a sensor.
    - threshold: float, the threshold for peak detection.
    - methods: list of strings, the transformation methods to apply (e.g., "rms").
    - outlier_method: str, the method to use for outlier detection ("zscore" or "iqr").
    - outlier_threshold: float, the threshold for detecting outliers.
    - attack_time_ms: int, attack time in milliseconds for envelope follower.
    - release_time_ms: int, release time in milliseconds for envelope follower.
    - v: bool, verbose output.
    Returns: tuple of (transformed_signal, peaks, timestamps_cleaned, transformed_signal, transformations).
    """

    transformations = sensor_data_transformations(data, 
                                methods=methods, 
                                outlier_method="zscore", 
                                outlier_threshold=100, 
                                attack_time_ms=attack_time_ms, 
                                release_time_ms=release_time_ms, 
                                v=v)
    
    timestamps_cleaned, transformed_signal = transformations[0]
    
    peaks = find_peaks(transformed_signal, threshold)
    peaks = [timestamps_cleaned[i] for i in peaks]

    return transformed_signal, peaks, timestamps_cleaned, transformed_signal, transformations

# https://kferg.dev/posts/2020/audio-reactive-programming-envelope-followers/
def envelope_follower(signal, sample_rate, attack_time_ms, release_time_ms):
    # Convert attack/release times to filter coefficients
    alpha_attack = np.exp(-1 / (sample_rate * attack_time_ms / 1000))
    alpha_release = np.exp(-1 / (sample_rate * release_time_ms / 1000))

    envelope = np.zeros_like(signal, dtype=np.double)
    current_envelope = 0.0

    for i in range(len(signal)):
        abs_sample = abs(signal[i])
        if abs_sample > current_envelope:
            current_envelope = alpha_attack * current_envelope + (1 - alpha_attack) * abs_sample
        else:
            current_envelope = alpha_release * current_envelope + (1 - alpha_release) * abs_sample
        envelope[i] = current_envelope
    return envelope


def process_sensor_data(
      filename, 
      methods=["rms"],
      threshold=0.3,
      attack_time_ms=5, 
      release_time_ms=500, 
      v=False, 
      plot=False):
  all_data, interrupt_timestamps = read_sensor_data_hex(filename, v=v)

  # get min of all timestamps in all_data
  global_min_time = min(min(sensor[0]) for sensor in all_data)

  if plot:
    p = bokeh.plotting.figure(
                title="Sensor Data and Energy Functions",
                x_axis_label="Time (s)",
                y_axis_label="Magnitude",
                width=1000,
                height=600,
            )
    colors = bokeh.palettes.Turbo256


  all_peaks = []
  all_sensor_data = []
  all_transformed_data = []
  all_timestamps = []
  for i, sensor_data in enumerate(all_data):

    transformations = sensor_data_transformations(sensor_data, 
                            methods=methods, 
                            outlier_method="zscore", 
                            outlier_threshold=100, 
                            attack_time_ms=attack_time_ms, 
                            release_time_ms=release_time_ms, 
                            v=v)
    
    timestamps_cleaned, transformed_signal = transformations[0]
    
    
    peaks = find_peaks(transformed_signal, threshold)
    peaks = [timestamps_cleaned[i] for i in peaks]

    all_peaks.append(peaks)
    all_transformed_data.append(transformed_signal)
    all_timestamps.append(timestamps_cleaned)
    all_sensor_data.append(transformed_signal)  # TODO: not the right thing

    if plot:
        # Plot sensor data
        for method, (timestamps, signal) in zip(methods, transformations):
          legend_label = f"Sensor {i + 1} - {method}"
          r = p.line(list(timestamps), signal, legend_label=legend_label, color=colors[i * 10 + 1])
          r.visible = False
  
  global_min_time = min(min(sensor) for sensor in all_timestamps)
  global_max_time = max(max(sensor) for sensor in all_timestamps)

  if plot: 
    p.legend.ncols = len(methods)
    p.legend.click_policy = "hide"
    p.add_layout(p.legend[0], 'right')
    bokeh.io.show(p)
    
  return interrupt_timestamps, all_sensor_data, all_transformed_data, all_peaks, all_timestamps, global_min_time, global_max_time



### audio synthesis

In [ ]:
def adsr(a_level, a_length, d_length, s_level, s_length, r_length):
  a = np.linspace(0, 1, a_length)
  d = np.linspace(1, s_level, d_length)[1:]
  s = np.full((s_length, ), s_level)
  r = np.linspace(s_level, 0, r_length)
  return np.concatenate([a, d, s, r])

def amp_envelope(adsr_env, start, total_length):
  if (start + len(adsr_env) >= total_length):
    return np.pad(adsr_env, (start, 0))[:-(start + len(adsr_env) - total_length)]
  else:
    return np.pad(adsr_env, (start, total_length - (len(adsr_env) + start)))

def playback(peaks, min_time, max_time):
  output_length = int(((max_time - min_time)) * 44100)
  output_signal = np.zeros(output_length)
  for i, peak_list in enumerate(peaks):
    freq_tone = librosa.tone(BALAFON_TUNING[i], sr=44100, length=output_length)
    freq_envelope = np.zeros(output_length)

    adsr_env = adsr(1, 1000, 10, 0.6, 2000, 3000)

    for peak in peak_list:
      amp_envelope1 = amp_envelope(adsr_env, int(((peak - min_time)) * 44100), output_length)
      freq_envelope = np.maximum(freq_envelope, amp_envelope1)

    output_signal += freq_envelope * freq_tone

  display(Audio(data=output_signal, rate=44100))
  return output_signal

In [ ]:
def generate_initial_audio_from_data(data_file, output_file, plot=False, v=False):
    _, _, _, all_peaks, _, min_time, max_time = process_sensor_data(data_file, threshold=0.2, v=v, plot=plot)
    output_signal = playback(all_peaks, min_time, max_time)
    sf.write(output_file, output_signal, 44100)
    

In [ ]:
generate_initial_audio_from_data("data/test2", "out/test2_out2.wav", plot=True)
# generate_initial_audio_from_data("data/test3-1", "out/test3-1_out2.wav", plot=True)

## Audio visualization

In [ ]:
def compute_energy_with_full_cqt(audio, center_freqs, sr=None, hop_length=512, bins_per_octave=20):
    """
    Compute energy functions using the full Constant-Q Transform (CQT) and map to specific frequencies.

    Parameters:
    - audio: audio signal as a numpy array
    - center_freqs: list of floats, specific center frequencies for the filterbank.
    - sr: sampling rate of the audio
    - hop_length: int, number of samples between successive frames for the CQT.
    - bins_per_octave: int, number of bins per octave in the CQT.

    Returns:
    - timestamps: numpy array, timestamps corresponding to the energy functions.
    - energies: list of numpy arrays, energy functions for each center frequency.
    """

    # Compute the full CQT
    cqt = librosa.cqt(audio, sr=sr, hop_length=hop_length, fmin=min(center_freqs), n_bins=bins_per_octave * int(np.ceil(np.log2(max(center_freqs) / min(center_freqs)))), bins_per_octave=bins_per_octave)

    # Get the frequencies corresponding to the CQT bins
    cqt_frequencies = librosa.cqt_frequencies(cqt.shape[0], fmin=min(center_freqs), bins_per_octave=bins_per_octave)

    # Map the desired frequencies to the closest CQT bins
    energies = []
    for freq in center_freqs:
        # Find the closest CQT bin
        closest_bin = np.argmin(np.abs(cqt_frequencies - freq))
        energy = np.abs(cqt[closest_bin])**2  # Energy is the squared magnitude
        energy -= np.median(energy)
        energy /= np.max(energy)
        energies.append(energy)
    
    # Compute timestamps for the energy function
    timestamps = np.arange(len(energies[0])) * hop_length / sr  # Energy timestamps in seconds

    return timestamps, energies

def plot_energy_functions_matplotlib(energies, energy_timestamps, sr, center_freqs):
    """
    Plot energy functions for each center frequency using Matplotlib.
    
    Parameters:
    - energies: list of numpy arrays, energy functions for each center frequency.
    - sr: int, sampling rate of the audio.
    - center_freqs: list of floats, center frequencies for the filterbank.
    """
    time = np.linspace(0, len(energies[0]) / sr, len(energies[0]))
    plt.figure(figsize=(12, 8))
    for i, energy in enumerate(energies):
        plt.plot(energy_timestamps, energy, label=f"{center_freqs[i]} Hz")
    plt.xlabel("Time (s)")
    plt.ylabel("Energy")
    plt.title("Energy Functions for Each Center Frequency")
    plt.legend()
    plt.show()

def process_audio_with_filterbank(audio_file, center_freqs, filter_size=500, plot=False):
    """
    Process an audio file with a filterbank and plot energy functions.
    
    Parameters:
    - audio_file: str, path to the audio file.
    - center_freqs: list of floats, center frequencies for the filterbank.
    - filter_size: int, size of the moving average filter to smooth the energy functions.
    - plot: bool, whether to plot the energy functions.
    """
    # Load the audio file
    audio, sr = librosa.load(audio_file, sr=None)
    
    # Filter the audio and compute energy functions
    energy_timestamps, energies = compute_energy_with_full_cqt(audio, center_freqs, sr)

    for i, energy in enumerate(energies):
        # moving average of the energy function
        energy = np.convolve(energy, np.ones(filter_size)/filter_size, mode='same')
        energies[i] = energy

    
    # Plot the energy functions
    if plot:
        plot_energy_functions_matplotlib(energies, energy_timestamps, sr, center_freqs)
    
    return energy_timestamps, energies


In [ ]:
energy_timestamps, energies = process_audio_with_filterbank("audio/ZOOM0004_INPUT1.WAV", BALAFON_TUNING, plot=True)